# 05 — Construction des variables et découpage anti-fuite

**Ce que fait ce notebook.** Il transforme le panel brut (`PANEL_SENTIMENT_WINDOWS_2020_2022.csv`) en un
jeu de données prêt pour la modélisation : variables normalisées, découpage temporel train/validation/test,
et batterie de tests automatiques garantissant l'absence de fuite d'information.

**Pourquoi c'est le notebook le plus important du mémoire.** Un modèle mal réglé donne un mauvais résultat,
qu'on voit et qu'on corrige. Une fuite d'information donne un **excellent** résultat, qu'on ne voit pas, et
qui s'effondre en soutenance quand quelqu'un pose la bonne question. Tout le travail de ce notebook consiste
à rendre la fuite **impossible par construction**, puis à le **prouver** par des tests.

## Plan

| § | Étape | Question |
|---|-------|----------|
| 1 | Chargement & inventaire | Quelles colonnes ai-je, et lesquelles sont légales ? |
| 2 | Liste blanche de features | Quelles fenêtres se ferment avant la cible ? |
| 3 | Normalisation intra-ticker | Comment comparer NVDA (μ=0.16) et TSLA (μ=0.044) ? |
| 4 | Variables de marché | Que faut-il contrôler pour ne pas confondre sentiment et momentum ? |
| 5 | Cibles | Que cherche-t-on exactement à prédire ? |
| 6 | Découpage temporel | Comment éviter d'entraîner sur le futur ? |
| 7 | Tests anti-fuite | Comment le **prouver** ? |
| 8 | Export | Que consomment les notebooks 06-08 ? |

In [1]:
# ============================================================================
#  CONFIGURATION — LA SEULE LIGNE À MODIFIER SI TU CHANGES DE MACHINE
# ============================================================================
PROJET = r"C:\Users\semy4\OneDrive\Bureau\Fintech_project"

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

DATA = os.path.join(PROJET, "data", "processed")
DOCS = os.path.join(PROJET, "docs")
os.makedirs(DOCS, exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 11
plt.style.use("seaborn-v0_8-whitegrid")

PANEL = os.path.join(DATA, "PANEL_SENTIMENT_WINDOWS_2020_2022.csv")
print("Panel :", PANEL)

Panel : C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\PANEL_SENTIMENT_WINDOWS_2020_2022.csv


---
## §1 — Chargement et inventaire des colonnes

Première étape : savoir exactement ce qu'on a. On classe chaque colonne selon la fenêtre temporelle à
laquelle elle appartient — c'est cette classification qui déterminera ce qui est utilisable.

In [2]:
df = pd.read_csv(PANEL)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print(f"Dimensions : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print(f"Période    : {df['Date'].min().date()} -> {df['Date'].max().date()}")
print(f"Tickers    : {sorted(df['Ticker'].unique())}\n")

# Classement des colonnes par suffixe de fenêtre
suffixes = ["_overnight", "_pre", "_night_full", "_open30", "_mkt", "_post"]
for s in suffixes:
    cols = [c_ for c_ in df.columns if c_.endswith(s)]
    print(f"{s:14s} : {len(cols):2d} colonnes -> {cols[:6]}{' ...' if len(cols) > 6 else ''}")

autres = [c_ for c_ in df.columns if not any(c_.endswith(s) for s in suffixes)]
print(f"\n{'autres':14s} : {len(autres):2d} colonnes -> {autres}")

print("\nValeurs manquantes (colonnes concernées) :")
na = df.isna().sum()
print(na[na > 0].sort_values(ascending=False).head(20))

Dimensions : 2,740 lignes x 76 colonnes
Période    : 2020-01-02 -> 2022-03-04
Tickers    : ['AAPL', 'AMZN', 'META', 'NVDA', 'TSLA']

_overnight     :  5 colonnes -> ['n_overnight', 'mu_overnight', 'sd_overnight', 'pos_overnight', 'neg_overnight']
_pre           :  7 colonnes -> ['n_pre', 'mu_pre', 'sd_pre', 'p10_pre', 'p90_pre', 'pos_pre'] ...
_night_full    :  8 colonnes -> ['n_night_full', 'mu_night_full', 'pos_night_full', 'neg_night_full', 'sd_night_full', 'nlog_night_full'] ...
_open30        : 10 colonnes -> ['n_open30', 'mu_open30', 'sd_open30', 'p10_open30', 'p90_open30', 'pos_open30'] ...
_mkt           : 10 colonnes -> ['n_mkt', 'mu_mkt', 'sd_mkt', 'p10_mkt', 'p90_mkt', 'pos_mkt'] ...
_post          : 10 colonnes -> ['n_post', 'mu_post', 'sd_post', 'p10_post', 'p90_post', 'pos_post'] ...

autres         : 26 colonnes -> ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker', 'prev_close', 'gap', 'ret_oc', 'ret_cc', 'prev_ret_cc', 'prev_volume', 'vol_20d', 'dmu_night', 'm

---
## §2 — La liste blanche : quelles variables ont le droit d'entrer dans un modèle

### Le principe, en une phrase

> Une variable est utilisable pour prédire une cible **si et seulement si** elle est entièrement connue
> **avant** que la cible ne commence à se former.

### Application à notre cas

La cible principale est `gap_t = Open_t / Close_{t−1} − 1`. Elle se forme **à 9h30 le jour J**.

Donc une variable est légale si elle est connue à 9h29 le jour J :

| Variable | Connue à | Légale pour `gap_t` ? |
|----------|----------|------------------------|
| `*_overnight` (16h J−1 → minuit) | minuit | ✅ |
| `*_pre` (minuit → 9h30) | 9h30 | ✅ (à la limite — voir encadré) |
| `*_night_full` (= overnight + pre) | 9h30 | ✅ |
| `Close_{t−1}`, `Volume_{t−1}` | 16h J−1 | ✅ |
| `*_open30` (9h30 → 10h) | 10h | ❌ le gap est déjà formé |
| `*_mkt` (10h → 16h) | 16h | ❌ |
| `*_post` (16h → minuit) | minuit J | ❌ |
| `Open_t`, `High_t`, `Low_t`, `Close_t` | pendant/après la séance | ❌ **(c'est la cible !)** |

> **Encadré — la nuance sur `pre`.** La fenêtre `pre` se ferme exactement à l'ouverture. En pratique,
> pour trader le gap il faut se positionner **avant** 9h30, donc il faut couper le flux quelques minutes
> plus tôt. On propose donc deux jeux de variables :
> - **jeu « réaliste »** : toutes les fenêtres nocturnes (`night_full`) — utilisé pour l'analyse ;
> - **jeu « conservateur »** : `overnight` seul (coupé à minuit) — utilisé comme **test de robustesse**.
>
> Si le signal survit au jeu conservateur, aucune objection de timing n'est possible. C'est ce qu'on
> vérifie au notebook 06.

On code cette règle plutôt que de l'appliquer à la main : une règle codée ne s'oublie pas.

In [3]:
# --------------------------------------------------------------------
# Règle formelle : heure (en h depuis minuit du jour J) à laquelle
# chaque famille de variables devient connue.
# --------------------------------------------------------------------
CONNUE_A = {
    "_overnight":   0.0,    # messages 16h(J-1) -> minuit : connus à minuit
    "_pre":         9.5,    # messages minuit -> 9h30
    "_night_full":  9.5,    # union des deux
    "_open30":     10.0,
    "_mkt":        16.0,
    "_post":       24.0,
}
DEBUT_CIBLE = {"gap": 9.5, "ret_oc": 9.5, "ret_cc": 9.5}   # tout démarre à l'ouverture

INTERDITES_TOUJOURS = ["Open", "High", "Low", "Close", "Adj Close", "Volume",
                       "gap", "ret_oc", "ret_cc", "y_gap", "y_oc", "y_cc",
                       "amplitude", "abs_gap", "regime"]

def est_legale(col, cible="gap", conservateur=False):
    """True si `col` peut servir à prédire `cible` sans fuite d'information."""
    if col in INTERDITES_TOUJOURS or col in ("Date", "Ticker"):
        return False
    for suf, h in CONNUE_A.items():
        if col.endswith(suf):
            if conservateur and suf in ("_pre", "_night_full"):
                return False                      # jeu conservateur : minuit seulement
            return h <= DEBUT_CIBLE[cible]
    return None                                   # colonne dérivée -> traitée au §4

legales   = [c_ for c_ in df.columns if est_legale(c_, "gap") is True]
illegales = [c_ for c_ in df.columns if est_legale(c_, "gap") is False]
a_traiter = [c_ for c_ in df.columns if est_legale(c_, "gap") is None]

print(f"LÉGALES pour prédire le gap ({len(legales)}) :")
for x in legales: print("   ", x)
print(f"\nINTERDITES ({len(illegales)}) : {illegales}")
print(f"\nÀ EXAMINER au cas par cas ({len(a_traiter)}) : {a_traiter}")

LÉGALES pour prédire le gap (20) :
    n_overnight
    mu_overnight
    sd_overnight
    pos_overnight
    neg_overnight
    n_pre
    mu_pre
    sd_pre
    p10_pre
    p90_pre
    pos_pre
    neg_pre
    n_night_full
    mu_night_full
    pos_night_full
    neg_night_full
    sd_night_full
    nlog_night_full
    nabn_night_full
    disp_night_full

INTERDITES (43) : ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker', 'gap', 'ret_oc', 'ret_cc', 'n_open30', 'mu_open30', 'sd_open30', 'p10_open30', 'p90_open30', 'pos_open30', 'neg_open30', 'n_mkt', 'mu_mkt', 'sd_mkt', 'p10_mkt', 'p90_mkt', 'pos_mkt', 'neg_mkt', 'n_post', 'mu_post', 'sd_post', 'p10_post', 'p90_post', 'pos_post', 'neg_post', 'nlog_open30', 'nabn_open30', 'disp_open30', 'nlog_mkt', 'nabn_mkt', 'disp_mkt', 'nlog_post', 'nabn_post', 'disp_post', 'y_gap', 'y_oc', 'y_cc']

À EXAMINER au cas par cas (13) : ['prev_close', 'prev_ret_cc', 'prev_volume', 'vol_20d', 'dmu_night', 'mu_night_ma3', 'mu_night_z20', 'mu_mkt_lag1',

---
## §3 — Normalisation intra-ticker : le z-score glissant

### Le problème, chiffré

| Ticker | `mu_night_full` moyen | Écart-type |
|--------|-----------------------|------------|
| NVDA   | 0.160 | 0.079 |
| AMZN   | 0.116 | 0.049 |
| META   | 0.112 | 0.068 |
| AAPL   | 0.093 | 0.042 |
| TSLA   | 0.044 | 0.032 |

Un sentiment de **0.12** est **médiocre** pour NVDA (en dessous de sa moyenne) et **exceptionnel** pour
TSLA (près de 3 écarts-types au-dessus). La valeur brute ne veut rien dire hors contexte.

Si on donne la valeur brute à un modèle, il apprend surtout **l'identité du ticker** : « quand `mu` est
élevé, c'est probablement NVDA, et NVDA a beaucoup monté en 2020-2021 ». C'est du bruit déguisé en signal,
et ça ne se généralisera pas.

### La solution : z-score sur fenêtre glissante

```
z_t = (x_t − moyenne(x sur les 60 jours PRÉCÉDENTS)) / écart-type(x sur les 60 jours PRÉCÉDENTS)
```

`z = +2` signifie alors : *« le sentiment d'aujourd'hui est 2 écarts-types au-dessus de ce qui est normal
pour cette action, ces 3 derniers mois »*. C'est comparable d'un ticker à l'autre, et c'est ça, le signal.

### Le détail qui fait tout : `.shift(1)`

```python
mu = s.rolling(60).mean().shift(1)      # <-- SANS ce shift, c'est de la fuite
```

Sans `.shift(1)`, la fenêtre de 60 jours **inclut le jour J lui-même**. On normaliserait donc `x_t` par une
moyenne qui contient déjà `x_t`. C'est une fuite : faible en apparence, mais elle suffit à gonfler
artificiellement toutes les performances. **C'est l'erreur n°1 en finance quantitative appliquée.**

### Pourquoi une fenêtre glissante plutôt qu'une moyenne globale

Une moyenne calculée sur toute la période 2020-2022 utiliserait des données de 2022 pour normaliser des
jours de 2020 — donc du futur. La fenêtre glissante n'utilise que le passé, à chaque date. Elle a en plus
l'avantage de s'adapter aux changements de régime (le niveau de bavardage n'est pas le même en mars 2020
et en janvier 2022).

In [4]:
FENETRE_Z = 60      # ~3 mois de bourse
MIN_OBS   = 20      # au moins 1 mois avant de produire une valeur

def zscore_glissant(s, fenetre=FENETRE_Z, minp=MIN_OBS):
    """z-score causal : n'utilise QUE le passé strict (d'où le .shift(1))."""
    mu = s.rolling(fenetre, min_periods=minp).mean().shift(1)
    sd = s.rolling(fenetre, min_periods=minp).std().shift(1)
    return (s - mu) / sd.replace(0, np.nan)

def rang_glissant(s, fenetre=252, minp=60):
    """Rang percentile du jour J parmi les `fenetre` jours précédents (0 à 1).

    Robuste aux valeurs extrêmes : contrairement au z-score, un jour aberrant
    ne déforme pas l'échelle. Utile car l'effet observé au notebook 04 vient
    surtout des QUEUES de distribution (Q1 et Q5).
    """
    return s.rolling(fenetre, min_periods=minp).apply(
        lambda w: (w[:-1] < w[-1]).mean(), raw=True)

A_NORMALISER = [c_ for c_ in ["mu_night_full", "mu_overnight", "mu_pre",
                              "sd_night_full", "sd_overnight", "sd_pre",
                              "nlog_night_full", "nabn_night_full",
                              "pos_rate_night_full", "neg_rate_night_full"]
                if c_ in df.columns]

print("Variables normalisées :")
for col in A_NORMALISER:
    df[f"z_{col}"] = df.groupby("Ticker")[col].transform(zscore_glissant)
    print(f"   z_{col}")

for col in ["mu_night_full", "nabn_night_full"]:
    if col in df.columns:
        df[f"rk_{col}"] = df.groupby("Ticker")[col].transform(rang_glissant)
        print(f"   rk_{col}  (rang percentile)")

# Contrôle : après normalisation, les tickers doivent avoir des moyennes proches de 0
print("\nAVANT normalisation (moyenne de mu_night_full par ticker) :")
print(df.groupby("Ticker")["mu_night_full"].mean().round(4).to_string())
print("\nAPRÈS normalisation (moyenne de z_mu_night_full par ticker) :")
print(df.groupby("Ticker")["z_mu_night_full"].mean().round(4).to_string())
print("\n-> Les écarts de niveau entre tickers ont disparu. C'est l'effet recherché.")

Variables normalisées :
   z_mu_night_full
   z_mu_overnight
   z_mu_pre
   z_sd_night_full
   z_sd_overnight
   z_sd_pre
   z_nlog_night_full
   z_nabn_night_full
   rk_mu_night_full  (rang percentile)
   rk_nabn_night_full  (rang percentile)

AVANT normalisation (moyenne de mu_night_full par ticker) :
Ticker
AAPL    0.0934
AMZN    0.1157
META    0.1116
NVDA    0.1600
TSLA    0.0444

APRÈS normalisation (moyenne de z_mu_night_full par ticker) :
Ticker
AAPL   -0.0102
AMZN   -0.0707
META   -0.1058
NVDA   -0.1173
TSLA   -0.0228

-> Les écarts de niveau entre tickers ont disparu. C'est l'effet recherché.


---
## §4 — Variables de contrôle de marché

### Pourquoi il en faut absolument

Imaginons qu'on trouve : *« sentiment nocturne élevé → gap positif »*. Un lecteur critique répondra
immédiatement :

> « Le sentiment est élevé **parce que** l'action a monté hier. Et une action qui a monté hier a tendance
> à ouvrir en hausse (momentum). Vous avez juste redécouvert le momentum, en passant par les tweets. »

C'est une objection **sérieuse** et il faut y répondre avec des données, pas avec des mots. La réponse
consiste à inclure des variables de momentum et de volatilité dans le modèle, puis à montrer que le
sentiment garde un pouvoir explicatif **une fois ces variables contrôlées**.

### Les contrôles retenus

| Variable | Définition | Ce qu'elle capture |
|----------|-----------|--------------------|
| `ret_cc_lag1` | rendement close-close de la veille | momentum court terme / réversion |
| `ret_oc_lag1` | séance de la veille | direction de fin de journée |
| `gap_lag1` | gap de la veille | persistance des gaps |
| `vol_20` | écart-type des rendements sur 20 j | régime de volatilité |
| `ampl_lag1` | amplitude (High−Low)/Open de la veille | agitation de la veille |
| `vol_dollar_z` | z-score du volume échangé | intensité de l'activité |
| `dow` | jour de la semaine | effets calendaires (lundi, vendredi) |

**Toutes sont décalées d'au moins un jour** (`shift(1)`) : elles ne contiennent donc que de l'information
disponible à la clôture de la veille.

In [5]:
g = df.groupby("Ticker")

df["ret_cc_lag1"] = g["ret_cc"].shift(1)
df["ret_oc_lag1"] = g["ret_oc"].shift(1)
df["gap_lag1"]    = g["gap"].shift(1)
df["gap_lag2"]    = g["gap"].shift(2)

# Volatilité réalisée sur 20 jours, calculée sur des rendements DÉJÀ décalés
df["vol_20"] = g["ret_cc"].transform(
    lambda s: s.shift(1).rolling(20, min_periods=10).std())

# Amplitude intraday de la veille
df["amplitude"] = (df["High"] - df["Low"]) / df["Open"]
df["ampl_lag1"] = g["amplitude"].shift(1)
df["ampl_ma5"]  = g["amplitude"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=3).mean())

# Volume en dollars, normalisé par ticker
df["vol_dollar"]   = df["Volume"] * df["Close"]
df["vol_dollar_l1"] = g["vol_dollar"].shift(1)
df["vol_dollar_z"] = df.groupby("Ticker")["vol_dollar_l1"].transform(
    lambda s: (s - s.rolling(60, min_periods=20).mean()) / s.rolling(60, min_periods=20).std())

# Effets calendaires
df["dow"] = df["Date"].dt.dayofweek          # 0 = lundi ... 4 = vendredi
df["mois"] = df["Date"].dt.month

CONTROLES = ["ret_cc_lag1", "ret_oc_lag1", "gap_lag1", "gap_lag2",
             "vol_20", "ampl_lag1", "ampl_ma5", "vol_dollar_z", "dow"]

print("Variables de contrôle créées :")
print(df[CONTROLES].describe().T[["count", "mean", "std", "min", "max"]].round(4))

print("\nVÉRIFICATION ANTI-FUITE — corrélation de chaque contrôle avec la cible du MÊME jour :")
for col in CONTROLES:
    s = df[[col, "gap"]].dropna()
    print(f"   {col:16s} rho(gap) = {s[col].corr(s['gap']):+.4f}")
print("""
Ces corrélations DOIVENT être faibles. Une valeur élevée signalerait qu'un
contrôle contient de l'information du jour J (donc une fuite).
""")

Variables de contrôle créées :
               count    mean     std     min     max
ret_cc_lag1   2735.0  0.0023  0.0317 -0.2639  0.1989
ret_oc_lag1   2735.0  0.0007  0.0242 -0.1279  0.1578
gap_lag1      2735.0  0.0016  0.0203 -0.2426  0.1320
gap_lag2      2730.0  0.0016  0.0203 -0.2426  0.1320
vol_20        2690.0  0.0276  0.0157  0.0086  0.1003
ampl_lag1     2735.0  0.0337  0.0228  0.0066  0.2496
ampl_ma5      2725.0  0.0337  0.0185  0.0092  0.1703
vol_dollar_z  2640.0  0.0886  1.1157 -2.0489  6.8045
dow           2740.0  2.0201  1.3942  0.0000  4.0000

VÉRIFICATION ANTI-FUITE — corrélation de chaque contrôle avec la cible du MÊME jour :
   ret_cc_lag1      rho(gap) = +0.0508
   ret_oc_lag1      rho(gap) = +0.1029
   gap_lag1         rho(gap) = -0.0461
   gap_lag2         rho(gap) = +0.0706
   vol_20           rho(gap) = +0.0516
   ampl_lag1        rho(gap) = -0.0021
   ampl_ma5         rho(gap) = +0.0533
   vol_dollar_z     rho(gap) = -0.0147
   dow              rho(gap) = -0.0186



---
## §5 — Les cibles

On définit trois problèmes distincts, qui font l'objet des notebooks 06, 07 et 08.

| Modèle | Cible | Type | Notebook | Attente |
|--------|-------|------|----------|---------|
| **M1** | `y_gap` = 1 si `gap > 0` | classification binaire | 06 | **signal présent** |
| **M2** | `y_oc` = 1 si `ret_oc > 0` | classification binaire | 07 | **pas de signal** (efficience) |
| **M3** | `log(amplitude)` | régression | 07 | **meilleur R²** des trois |

### Deux raffinements importants

**a) Le gap « épuré » (`gap_excess`).** Une partie du gap est un simple mouvement de marché : si le
Nasdaq ouvre en hausse, les 5 actions ouvrent en hausse. Prédire ça, c'est prédire le marché, pas
l'action. On construit donc `gap_excess = gap_ticker − moyenne des gaps du jour sur les 5 tickers`.
Si le sentiment prédit encore `gap_excess`, alors le signal est **spécifique à l'action** — un résultat
bien plus fort.

**b) La zone morte (`y_gap_net`).** Un gap de +0.01 % n'est pas exploitable : les frais le mangent. On
définit une cible qui **ignore** les gaps trop petits pour être tradés (|gap| < 15 points de base). C'est
elle qui compte pour le backtest du notebook 08.

In [6]:
# a) Gap en excès du marché (moyenne transversale du jour)
df["gap_mkt"]    = df.groupby("Date")["gap"].transform("mean")
df["gap_excess"] = df["gap"] - df["gap_mkt"]
df["y_gap_excess"] = (df["gap_excess"] > 0).astype(int)

# b) Zone morte : on ignore les gaps trop petits pour couvrir les coûts
SEUIL_BP = 15                       # 15 points de base = 0,15 %
seuil = SEUIL_BP / 10_000
df["y_gap_net"] = np.where(df["gap"] > seuil, 1,
                    np.where(df["gap"] < -seuil, 0, np.nan))

# c) Cible de risque
df["log_ampl"] = np.log(df["amplitude"].clip(lower=1e-6))

print("Équilibre des classes (proportion de 1) :")
for c_ in ["y_gap", "y_oc", "y_cc", "y_gap_excess", "y_gap_net"]:
    s = df[c_].dropna()
    print(f"   {c_:14s} : {s.mean():.3f}   (n = {len(s):,})")

print(f"\nJours neutralisés par la zone morte de {SEUIL_BP} bp : "
      f"{df['y_gap_net'].isna().sum() - df['gap'].isna().sum():,} "
      f"({(df['y_gap_net'].isna().mean())*100:.1f} % du total)")

print("""
LECTURE : y_gap est autour de 0.55-0.58, pas 0.50. Le marché monte plus
souvent qu'il ne baisse sur 2020-2022. Un modèle qui prédit TOUJOURS "hausse"
obtiendrait donc ~56 % d'exactitude sans aucune intelligence.
=> L'exactitude (accuracy) est une métrique TROMPEUSE ici.
=> On utilisera l'AUC, le MCC et le Brier score (notebook 06).
""")

Équilibre des classes (proportion de 1) :
   y_gap          : 0.589   (n = 2,740)
   y_oc           : 0.505   (n = 2,740)
   y_cc           : 0.534   (n = 2,740)
   y_gap_excess   : 0.472   (n = 2,740)
   y_gap_net      : 0.599   (n = 2,358)

Jours neutralisés par la zone morte de 15 bp : 382 (13.9 % du total)

LECTURE : y_gap est autour de 0.55-0.58, pas 0.50. Le marché monte plus
souvent qu'il ne baisse sur 2020-2022. Un modèle qui prédit TOUJOURS "hausse"
obtiendrait donc ~56 % d'exactitude sans aucune intelligence.
=> L'exactitude (accuracy) est une métrique TROMPEUSE ici.
=> On utilisera l'AUC, le MCC et le Brier score (notebook 06).



---
## §6 — Le découpage temporel

### La règle absolue

> **Jamais** de `train_test_split(shuffle=True)` sur des séries temporelles.

Un découpage aléatoire met des jours de 2022 dans l'entraînement et des jours de 2020 dans le test. Le
modèle apprend alors le futur pour prédire le passé. Les scores obtenus sont excellents et **entièrement
faux**. C'est la faute la plus fréquente et la plus rédhibitoire dans ce type de mémoire.

### Le découpage retenu

```
├──────────── TRAIN ────────────┼──── VALID ────┼──── TEST ────┤
2020-01-02              2021-06-30      2021-11-30      2022-03-04
       ~370 jours              ~105 jours       ~65 jours
```

- **TRAIN** — on ajuste les paramètres des modèles.
- **VALIDATION** — on choisit les hyperparamètres, les seuils, le modèle final. On peut y revenir autant
  qu'on veut.
- **TEST** — on ne le regarde **qu'une seule fois**, à la toute fin. Chaque coup d'œil supplémentaire
  transforme le test en validation et invalide le chiffre annoncé.

### Le purge gap

Entre chaque bloc, on retire quelques jours. Pourquoi : les features utilisent des moyennes mobiles sur
60 jours. Le dernier jour du train et le premier jour de la validation partagent donc une partie de leur
historique. Un embargo de 5 jours coupe ce chevauchement (López de Prado, *Advances in Financial Machine
Learning*, 2018).

### Et le walk-forward

Pour les résultats du mémoire, on complète par une **validation glissante** : on entraîne sur une fenêtre,
on teste sur la période suivante, on avance, on recommence. On obtient ainsi plusieurs mesures de
performance hors échantillon au lieu d'une seule — donc une idée de la **stabilité** du signal, et pas
seulement de son niveau.

In [7]:
FIN_TRAIN  = pd.Timestamp("2021-06-30")
FIN_VALID  = pd.Timestamp("2021-11-30")
PURGE_JOURS = 5

def assigner_bloc(d):
    if d <= FIN_TRAIN:                                        return "train"
    if d <= FIN_TRAIN + pd.Timedelta(days=PURGE_JOURS):       return "purge"
    if d <= FIN_VALID:                                        return "valid"
    if d <= FIN_VALID + pd.Timedelta(days=PURGE_JOURS):       return "purge"
    return "test"

df["bloc"] = df["Date"].apply(assigner_bloc)

resume = df.groupby("bloc").agg(
    n_lignes=("Date", "size"),
    n_jours=("Date", "nunique"),
    debut=("Date", "min"),
    fin=("Date", "max"),
    pct_gap_pos=("y_gap", "mean"),
).reindex(["train", "purge", "valid", "test"])
resume["pct_gap_pos"] = (resume["pct_gap_pos"] * 100).round(1)
print(resume.to_string())

print("""
CONTRÔLE À FAIRE : la colonne pct_gap_pos doit être comparable entre blocs.
Si le train est à 60 % de gaps positifs et le test à 45 %, le test tombe dans
un régime de marché différent : les performances y seront mauvaises pour des
raisons qui n'ont rien à voir avec la qualité du modèle. Il faut alors le dire
explicitement dans le mémoire plutôt que de le subir.
""")

# --- Générateur walk-forward (utilisé aux notebooks 06 et 07) ---
def splits_walk_forward(dates, n_plis=5, taille_test=60, min_train=250):
    """Découpage glissant à fenêtre EXTENSIBLE (expanding window).

    Renvoie une liste de (dates_train, dates_test) strictement ordonnées
    dans le temps, avec un embargo de PURGE_JOURS entre les deux.
    """
    jours = np.sort(pd.unique(dates))
    plis, fin = [], len(jours)
    for _ in range(n_plis):
        d0_test = fin - taille_test
        if d0_test - PURGE_JOURS < min_train:
            break
        plis.append((jours[:d0_test - PURGE_JOURS], jours[d0_test:fin]))
        fin = d0_test
    return plis[::-1]

plis = splits_walk_forward(df["Date"], n_plis=5, taille_test=60)
print(f"\nWalk-forward : {len(plis)} plis")
for i, (tr, te) in enumerate(plis, 1):
    print(f"  Pli {i} : train {pd.Timestamp(tr[0]).date()} -> {pd.Timestamp(tr[-1]).date()} "
          f"({len(tr):3d} j)  |  test {pd.Timestamp(te[0]).date()} -> {pd.Timestamp(te[-1]).date()} ({len(te)} j)")

       n_lignes  n_jours      debut        fin  pct_gap_pos
bloc                                                       
train      1885      377 2020-01-02 2021-06-30         59.5
purge        25        5 2021-07-01 2021-12-03         76.0
valid       520      104 2021-07-06 2021-11-30         60.6
test        310       62 2021-12-06 2022-03-04         51.0

CONTRÔLE À FAIRE : la colonne pct_gap_pos doit être comparable entre blocs.
Si le train est à 60 % de gaps positifs et le test à 45 %, le test tombe dans
un régime de marché différent : les performances y seront mauvaises pour des
raisons qui n'ont rien à voir avec la qualité du modèle. Il faut alors le dire
explicitement dans le mémoire plutôt que de le subir.


Walk-forward : 4 plis
  Pli 1 : train 2020-01-02 -> 2021-03-16 (303 j)  |  test 2021-03-24 -> 2021-06-17 (60 j)
  Pli 2 : train 2020-01-02 -> 2021-06-10 (363 j)  |  test 2021-06-18 -> 2021-09-13 (60 j)
  Pli 3 : train 2020-01-02 -> 2021-09-03 (423 j)  |  test 2021-09-14 ->

---
## §7 — Les tests anti-fuite

C'est la section à montrer en soutenance. Quatre tests automatiques ; si l'un échoue, on ne modélise pas.

| Test | Ce qu'il vérifie | Comment il échoue |
|------|------------------|-------------------|
| **T1** | Aucune feature ne corrèle anormalement avec la cible du jour | ρ > 0.4 → la feature contient la cible |
| **T2** | Ordre chronologique strict entre les blocs | max(train) ≥ min(test) → chevauchement |
| **T3** | Le sentiment **futur** ne prédit pas le passé | ρ(lag −1) ≈ ρ(lag 0) → désalignement |
| **T4** | Une cible **permutée** ne se prédit pas | AUC > 0.55 sur du bruit → fuite dans le pipeline |

In [8]:
FEATURES_SENT = [c_ for c_ in df.columns
                 if c_.startswith(("z_", "rk_")) and "night" in c_ or c_.startswith("z_mu_overnight")]
FEATURES_SENT = sorted(set([c_ for c_ in df.columns if c_.startswith(("z_", "rk_"))]))
FEATURES = FEATURES_SENT + CONTROLES
FEATURES = [c_ for c_ in FEATURES if df[c_].notna().sum() > 500]

print(f"{len(FEATURES)} variables retenues :\n  " + "\n  ".join(FEATURES))

echecs = []

# ---------------- T1 : corrélation anormale ----------------
print("\n" + "=" * 70 + "\nT1 — Corrélation feature <-> cible du même jour\n" + "=" * 70)
SEUIL_T1 = 0.40
for f in FEATURES:
    s = df[[f, "gap"]].dropna()
    if len(s) < 100:
        continue
    r_ = abs(s[f].corr(s["gap"]))
    if r_ > SEUIL_T1:
        echecs.append(f"T1 : {f} corrèle à {r_:.3f} avec gap")
        print(f"  [ÉCHEC] {f:22s} |rho| = {r_:.3f}")
print("  [OK] Aucune corrélation suspecte." if not echecs else "")

# ---------------- T2 : ordre chronologique ----------------
print("\n" + "=" * 70 + "\nT2 — Ordre chronologique des blocs\n" + "=" * 70)
for a, b in [("train", "valid"), ("valid", "test")]:
    fin_a = df.loc[df["bloc"] == a, "Date"].max()
    deb_b = df.loc[df["bloc"] == b, "Date"].min()
    ok = fin_a < deb_b
    print(f"  max({a}) = {fin_a.date()}  <  min({b}) = {deb_b.date()}   -> {'OK' if ok else 'ÉCHEC'}")
    if not ok:
        echecs.append(f"T2 : chevauchement {a}/{b}")

# ---------------- T3 : lags négatifs ----------------
print("\n" + "=" * 70 + "\nT3 — Le sentiment FUTUR prédit-il le gap d'aujourd'hui ?\n" + "=" * 70)
tmp = df.copy()
tmp["z_futur"] = tmp.groupby("Ticker")["z_mu_night_full"].shift(-1)
s0 = tmp[["z_mu_night_full", "gap"]].dropna()
s1 = tmp[["z_futur", "gap"]].dropna()
r0 = s0["z_mu_night_full"].corr(s0["gap"])
r1 = s1["z_futur"].corr(s1["gap"])
print(f"  rho(sentiment de J   -> gap de J) = {r0:+.4f}   <- le signal")
print(f"  rho(sentiment de J+1 -> gap de J) = {r1:+.4f}   <- doit être proche de 0")
if abs(r1) > 0.5 * abs(r0):
    echecs.append("T3 : le sentiment futur prédit le passé -> désalignement temporel")
    print("  [ÉCHEC] Alignement des fenêtres à revoir.")
else:
    print("  [OK] Le signal est bien orienté dans le temps.")

# ---------------- T4 : cible permutée ----------------
print("\n" + "=" * 70 + "\nT4 — Test placebo : cible aléatoire\n" + "=" * 70)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

d = df.dropna(subset=["y_gap"]).copy()
tr, te = d[d["bloc"] == "train"], d[d["bloc"] == "valid"]
pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                     LogisticRegression(max_iter=1000, C=1.0))

rng = np.random.default_rng(0)
y_bidon = rng.permutation(tr["y_gap"].values)
pipe.fit(tr[FEATURES], y_bidon)
auc_bidon = roc_auc_score(te["y_gap"], pipe.predict_proba(te[FEATURES])[:, 1])

pipe.fit(tr[FEATURES], tr["y_gap"])
auc_vrai = roc_auc_score(te["y_gap"], pipe.predict_proba(te[FEATURES])[:, 1])

print(f"  AUC avec cible PERMUTÉE : {auc_bidon:.4f}   (doit être ~0.50)")
print(f"  AUC avec cible RÉELLE   : {auc_vrai:.4f}   (doit être nettement > 0.50)")
if abs(auc_bidon - 0.5) > 0.05:
    echecs.append("T4 : le pipeline apprend du bruit -> fuite structurelle")
    print("  [ÉCHEC]")
else:
    print("  [OK] Le pipeline n'apprend rien sur du bruit : pas de fuite structurelle.")

print("\n" + "=" * 70)
print("AUCUNE FUITE DÉTECTÉE — on peut modéliser." if not echecs
      else "ÉCHECS :\n  - " + "\n  - ".join(echecs))
print("=" * 70)

19 variables retenues :
  rk_mu_night_full
  rk_nabn_night_full
  z_mu_night_full
  z_mu_overnight
  z_mu_pre
  z_nabn_night_full
  z_nlog_night_full
  z_sd_night_full
  z_sd_overnight
  z_sd_pre
  ret_cc_lag1
  ret_oc_lag1
  gap_lag1
  gap_lag2
  vol_20
  ampl_lag1
  ampl_ma5
  vol_dollar_z
  dow

T1 — Corrélation feature <-> cible du même jour
  [OK] Aucune corrélation suspecte.

T2 — Ordre chronologique des blocs
  max(train) = 2021-06-30  <  min(valid) = 2021-07-06   -> OK
  max(valid) = 2021-11-30  <  min(test) = 2021-12-06   -> OK

T3 — Le sentiment FUTUR prédit-il le gap d'aujourd'hui ?
  rho(sentiment de J   -> gap de J) = +0.2656   <- le signal
  rho(sentiment de J+1 -> gap de J) = +0.1036   <- doit être proche de 0
  [OK] Le signal est bien orienté dans le temps.

T4 — Test placebo : cible aléatoire
  AUC avec cible PERMUTÉE : 0.4796   (doit être ~0.50)
  AUC avec cible RÉELLE   : 0.6522   (doit être nettement > 0.50)
  [OK] Le pipeline n'apprend rien sur du bruit : pas de fu

---
## §8 — Export

On sauvegarde le jeu enrichi ainsi qu'un petit fichier de configuration lu par les notebooks 06 à 08.
Centraliser la liste des features et les dates de découpage évite qu'ils divergent d'un notebook à
l'autre — une source classique de résultats non reproductibles.

In [9]:
import json

CIBLES = ["y_gap", "y_oc", "y_cc", "y_gap_excess", "y_gap_net",
          "gap", "ret_oc", "ret_cc", "gap_excess", "log_ampl", "amplitude"]
META = ["Date", "Ticker", "bloc", "Open", "Close", "High", "Low", "Volume"]

# On conserve aussi les variables BRUTES des fenêtres prédictives : elles ne servent
# pas aux modèles (qui utilisent les versions normalisées z_/rk_) mais les notebooks
# 07 et 09 en ont besoin pour les analyses descriptives et les figures.
BRUTES = [c_ for c_ in df.columns
          if c_.endswith(("_overnight", "_pre", "_night_full")) and not c_.startswith(("z_", "rk_"))]

garder = META + sorted(set(FEATURES)) + BRUTES + [c_ for c_ in CIBLES if c_ in df.columns]
garder = [c_ for c_ in dict.fromkeys(garder) if c_ in df.columns]

out = df[garder].copy()
chemin = os.path.join(DATA, "DATASET_MODELISATION_2020_2022.csv")
out.to_csv(chemin, index=False)

# Jeu conservateur : uniquement les fenêtres closes à minuit
FEATURES_CONS = [f for f in FEATURES if "_pre" not in f and "night_full" not in f]

config = {
    "features": FEATURES,
    "features_conservateur": FEATURES_CONS,
    "features_sentiment": [f for f in FEATURES if f.startswith(("z_", "rk_"))],
    "controles": CONTROLES,
    "fin_train": str(FIN_TRAIN.date()),
    "fin_valid": str(FIN_VALID.date()),
    "purge_jours": PURGE_JOURS,
    "seuil_zone_morte_bp": SEUIL_BP,
    "fenetre_zscore": FENETRE_Z,
}
with open(os.path.join(DATA, "config_modelisation.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"Écrit : {chemin}")
print(f"        dont {len(BRUTES)} colonnes brutes conservées pour l'analyse descriptive")
print(f"        {out.shape[0]:,} lignes x {out.shape[1]} colonnes")
print(f"Écrit : config_modelisation.json  ({len(FEATURES)} features)")
print("\nRépartition par bloc :")
print(out["bloc"].value_counts().reindex(["train", "purge", "valid", "test"]).to_string())
print("\nComplétude des features (train uniquement) :")
tr_ = out[out["bloc"] == "train"]
print((tr_[FEATURES].notna().mean() * 100).round(1).sort_values().head(10).to_string())

Écrit : C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\DATASET_MODELISATION_2020_2022.csv
        dont 20 colonnes brutes conservées pour l'analyse descriptive
        2,740 lignes x 58 colonnes
Écrit : config_modelisation.json  (19 features)

Répartition par bloc :
bloc
train    1885
purge      25
valid     520
test      310

Complétude des features (train uniquement) :
rk_mu_night_full      84.4
rk_nabn_night_full    84.4
z_mu_night_full       94.7
z_mu_overnight        94.7
z_mu_pre              94.7
z_nabn_night_full     94.7
z_nlog_night_full     94.7
z_sd_night_full       94.7
z_sd_overnight        94.7
z_sd_pre              94.7


---
## §9 — Ce qu'il faut retenir de ce notebook

1. **Une variable n'est utilisable que si elle est connue avant la cible.** Cette règle est codée
   (`est_legale`), pas appliquée à la main — donc elle ne peut pas être oubliée.
2. **La normalisation intra-ticker n'est pas cosmétique.** Sans elle, le modèle apprend l'identité du
   ticker plutôt que le signal. Et elle doit être **causale** (`.shift(1)`).
3. **Les contrôles de marché sont là pour répondre à une objection**, pas pour améliorer le score : ils
   permettent de dire « le sentiment apporte quelque chose **en plus** du momentum ».
4. **Le découpage est temporel, avec embargo.** Le test est verrouillé.
5. **Quatre tests prouvent l'absence de fuite.** Le test placebo (T4) est le plus convaincant : si le
   pipeline apprend quelque chose sur une cible aléatoire, tout le reste est faux.

Le notebook 06 peut maintenant entraîner les modèles sur `DATASET_MODELISATION_2020_2022.csv`.